# Jordan Sensitivity Toy Model

Chapter 9: Matrix Decompositions and Beyond · *Mathematics for AI and Machine Learning* by Fuheng Wu

Show how nearly defective matrices react strongly to small perturbations.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-9/jordan-sensitivity/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import math

import torch

# The demo's default sliders: center eigenvalue and perturbation size
lam, eps = 1.2, 0.03


def perturbed(lam, e):
    """At e = 0 this is the Jordan block [[lam, 1], [0, lam]] (defective)."""
    return torch.tensor([[lam + e, 1.0],
                         [e, lam - e]], dtype=torch.float64)


A = perturbed(lam, eps)
ev = torch.linalg.eigvals(A).real.sort(descending=True).values
print("eigenvalues =", ev.tolist(), " gap =", (ev[0] - ev[1]).item())
print("predicted gap 2 sqrt(eps^2 + eps) =", 2 * math.sqrt(eps ** 2 + eps))

# ── A perturbation of size eps moves the eigenvalues by about sqrt(eps) ─────
print("\n   eps        gap       gap / eps    gap / sqrt(eps)")
for e in (1e-1, 1e-2, 1e-4, 1e-6, 1e-8):
    ev = torch.linalg.eigvals(perturbed(lam, e)).real
    gap = (ev.max() - ev.min()).item()
    print(f"{e:8.0e}  {gap:.3e}  {gap / e:10.1f}   {gap / math.sqrt(e):.4f}")

# ── A diagonalizable matrix moves only by about eps ─────────────────────────
D = torch.tensor([[lam, 0.0], [0.0, lam + 0.5]], dtype=torch.float64)
E = torch.tensor([[1e-6, 0.0], [1e-6, -1e-6]], dtype=torch.float64)
shift = (torch.linalg.eigvals(D + E).real.sort().values - torch.tensor([lam, lam + 0.5], dtype=torch.float64)).abs().max()
print("\ndiagonalizable matrix, perturbation 1e-6: eigenvalues move by", shift.item())

# At eps = 0 only one independent eigenvector exists
J = perturbed(lam, 0.0)
print("Jordan block: rank(J - lam I) =", torch.linalg.matrix_rank(J - lam * torch.eye(2, dtype=J.dtype)).item(),
      "-> one eigenvector for a double eigenvalue")

eigenvalues = [1.3757839583124694, 1.0242160416875306]  gap = 0.3515679166249388
predicted gap 2 sqrt(eps^2 + eps) = 0.3515679166249389

   eps        gap       gap / eps    gap / sqrt(eps)
   1e-01  6.633e-01         6.6   2.0976
   1e-02  2.010e-01        20.1   2.0100
   1e-04  2.000e-02       200.0   2.0001
   1e-06  2.000e-03      2000.0   2.0000
   1e-08  2.000e-04     20000.0   2.0000

diagonalizable matrix, perturbation 1e-6: eigenvalues move by 9.999999999177334e-07
Jordan block: rank(J - lam I) = 1 -> one eigenvector for a double eigenvalue
